<a href="https://colab.research.google.com/github/nickdhollman/Python-Projects/blob/BAN-5753-Advanced-Analytics/BAN5753_Exercise3_Ensemble_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BAN5753 Exercise 3
# Nick Hollman

In [ ]:
import sys
import numpy as np
import scipy
import matplotlib
import seaborn as sns

print(sys.executable)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)

In [ ]:
# Install Feature-engine - https://feature-engine.trainindata.com/en/latest/user_guide/creation/CyclicalFeatures.html
%pip install feature-engine

In [ ]:
#Install Category Encoding package
%pip install category_encoders

In [ ]:
# Install SHAP (SHapley Additive exPlanations) package.
# SHAP is a powerful tool used to explain the predictions made by machine learning models.
%pip install shap

In [ ]:
#Import Libraries
import pandas as pd
import os
import numpy as np
from numpy import mean
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')
from pylab import rcParams
#rcParams['figure.figsize']=16,15
rcParams.update({'font.size': 14})
# pd.options.display.max_columns = 25
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from numpy import argmax
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_predict # Added for model tuning and training-based threshold selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from category_encoders.target_encoder import TargetEncoder
from category_encoders.ordinal import OrdinalEncoder
from category_encoders.woe import WOEEncoder
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve, average_precision_score, precision_recall_curve, recall_score, precision_score, f1_score # Added for threshold selection and evaluation metrics
from feature_engine.creation import CyclicalFeatures
from sklearn.impute import SimpleImputer # for imputation

In [ ]:
#import data into colab
from google.colab import files

uploaded = files.upload()

In [ ]:
admission = pd.read_csv("hospital_admissions_with_missing.csv")

### Data Preprocessing

In [ ]:
# View the top of the data
admission.head()

In [ ]:
# View the share of the data (columns, rows)
print(admission.shape)

In [ ]:
# Variable types
admission.info()

In [ ]:
# Columns
admission.columns

#### 2a Remove variables that are excluded from the analysis and document the resulting predictor set

In [ ]:
# Create list of variables to remove
# FOLLOWUP_EMAILS is also excluded because it is measured after the hospital visit
# and would not be available at the time the admission prediction is made.

exclude_list = [
    'Visit_Year',
    'Patient_ID',
    'PRIOR_PROVIDER_TYPE',
    'TOTAL',
    'AllocProportion',
    'SampleSize',
    'ActualProportion',
    'SelectionProb',
    'SamplingWeight',
    'FOLLOWUP_EMAILS'
]

admission_model = admission.drop(columns=exclude_list)

In [ ]:
admission_model = admission.drop(columns=exclude_list)
#verify
print(admission_model.shape)
admission_model.columns

#### 2b - Create defensible cyclical representations of Visit_Date and Visit_Month,explaining the period and transformation used.

In [ ]:
# inspect both variables
print(admission_model['Visit_Date'].unique())
print(admission_model['Visit_Month'].unique())

In [ ]:
# convert month to numerical labels
month_map = {
    'Jan': 1,
    'Feb': 2,
    'Mar': 3,
    'Apr': 4,
    'May': 5,
    'Jun': 6,
    'Jul': 7,
    'Aug': 8,
    'Sep': 9,
    'Oct': 10,
    'Nov': 11,
    'Dec': 12
}

admission_model['Visit_Month_Num'] = admission_model['Visit_Month'].map(month_map)

In [ ]:
# verify
admission_model[['Visit_Month', 'Visit_Month_Num']].drop_duplicates().sort_values('Visit_Month_Num').reset_index(drop=True)

In [ ]:
# Manual transformation - Month
admission_model['Visit_Month_sin'] = np.sin(
    2 * np.pi * admission_model['Visit_Month_Num'] / 12
)

admission_model['Visit_Month_cos'] = np.cos(
    2 * np.pi * admission_model['Visit_Month_Num'] / 12
)

In [ ]:
# Manual transformation - Date
admission_model['Visit_Date_sin'] = np.sin(
    2 * np.pi * admission_model['Visit_Date'] / 31
)

admission_model['Visit_Date_cos'] = np.cos(
    2 * np.pi * admission_model['Visit_Date'] / 31
)

In [ ]:
# Subset manually created data
cyclical_check = admission_model[
    ['Visit_Date', 'Visit_Month_Num']
].copy()

In [ ]:
# Create features from CyclicalFeatures
cyclical = CyclicalFeatures(
    variables=['Visit_Date', 'Visit_Month_Num'],
    max_values={
        'Visit_Date': 31,
        'Visit_Month_Num': 12
    },
    drop_original=False
)

cyclical_check = cyclical.fit_transform(cyclical_check)

In [ ]:
cyclical_check.head()

In [ ]:
print(admission_model.columns.tolist())

In [ ]:
# Test if manual variables == varaibles created from CyclicalFeatures
print(
    np.allclose(
        admission_model['Visit_Date_sin'],
        cyclical_check['Visit_Date_sin']
    )
)

print(
    np.allclose(
        admission_model['Visit_Date_cos'],
        cyclical_check['Visit_Date_cos']
    )
)

print(
    np.allclose(
        admission_model['Visit_Month_sin'],
        cyclical_check['Visit_Month_Num_sin']
    )
)

print(
    np.allclose(
        admission_model['Visit_Month_cos'],
        cyclical_check['Visit_Month_Num_cos']
    )
)

#### Remove Original Date and Month Variables

After creating and verifying the cyclical representations of Visit_Date and Visit_Month, the original Visit_Date, Visit_Month, and temporary Visit_Month_Num variables were removed from the predictor set. This avoids representing the same time-related information in both its original and cyclical forms. The models will therefore use the cyclical variables created for the assignment.

In [ ]:
# Remove original date and month variables after creating cyclical features
date_variables_to_remove = [
    'Visit_Date',
    'Visit_Month',
    'Visit_Month_Num'
]

admission_model = admission_model.drop(columns=date_variables_to_remove)

In [ ]:
# Verify original variables were removed
print("Original date/month variables remaining:")
print([
    col for col in date_variables_to_remove
    if col in admission_model.columns
])

# Verify cyclical variables are still present
cyclical_variables = [
    'Visit_Date_sin',
    'Visit_Date_cos',
    'Visit_Month_sin',
    'Visit_Month_cos'
]

print("\nCyclical variables present:")
print([
    col for col in cyclical_variables
    if col in admission_model.columns
])

# Verify resulting dataset dimensions
print("\nDataset shape:", admission_model.shape)

#### 2c - Convert Local_Resident to a model-compatible representation and verify the resulting values.

In [ ]:
# descriptive stats for local_resident
admission_model['Local_Resident'].value_counts(dropna=False)

In [ ]:
# unique values
admission_model['Local_Resident'].unique()

In [ ]:
# map local_resident to numerical values
admission_model['Local_Resident'] = admission_model['Local_Resident'].map({
    'Y': 1,
    'N': 0
})

In [ ]:
# verify it was successful
print(admission_model['Local_Resident'].value_counts(dropna=False))
print(admission_model['Local_Resident'].unique())

### 2d) Extra credit: If you use hospital_admissions_with_missing.csv, identify the missingness pattern, select and justify an appropriate treatment, and fit that treatment using training data only

In [ ]:
# Get an understanding of the prevalance of missing data
missing_summary = pd.DataFrame({
    'Missing_Count': admission_model.isnull().sum(),
    'Missing_Percent': admission_model.isnull().mean() * 100
})

missing_summary = missing_summary[
    missing_summary['Missing_Count'] > 0
]

missing_summary

In [ ]:
# Missing value proportion by target class - restrict to only the variables above identified to have missing data
missing_vars = [
    'avg_income',
    'REFERRAL_SOURCE',
    'ETHNICITY',
    'BMI',
    'OXYGEN_LEVEL'
]

missing_by_target = (
    admission_model
    .groupby('Admitted')[missing_vars]
    .apply(lambda x: x.isna().mean() * 100)
)

missing_by_target

#### Missingness was low across the five variables with missing values, ranging from 1.2% to 1.5%. Missingness was generally similar between admitted and non-admitted patients, although ETHNICITY showed a larger difference between the two groups. This is a potential limitation. Numeric variables will be imputed using the median, while categorical variables will be imputed using the most frequent value. The imputation will be performed after the training-validation split below so that imputation values are fit using only the training data to prevent data leakage.

### 3a) Create a reproducible 70/30 training and validation split.

In [ ]:
# proceed with split and imputation
# Separate predictors and target
X = admission_model.drop(columns=['Admitted'])
y = admission_model['Admitted']

In [ ]:
# Check target distribution
print(y.value_counts())
print(y.value_counts(normalize=True))

### 3b) Preserve Target Distribution

A stratified 70/30 training-validation split will be used to preserve the distribution of the target variable (`Admitted`). All preprocessing decisions, encoder fitting, feature selection, and hyperparameter tuning will use only the training partition.

### 3c) Random Seed and Data Leakage Prevention

A random seed of 42 will be used for reproducibility. The validation partition will remain separate during model development. Preprocessing and model tuning will be fit using only the training data and then applied to the validation data to prevent information from the validation partition from influencing model development.

In [ ]:
# Create 70/30 training and validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30, # validation size 30%
    random_state=42, # specify seed for reproducability
    stratify=y # stratify split by target
)

In [ ]:
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True))

In [ ]:
# subset variables with missing data for imputation
numeric_missing = [
    'avg_income',
    'BMI',
    'OXYGEN_LEVEL'
]

categorical_missing = [
    'REFERRAL_SOURCE',
    'ETHNICITY'
]

#### Missing Data Imputation

Median imputation was used for the numeric variables because it is less influenced by extreme values than the mean. Mode imputation was used for the categorical variables because it replaces missing values with the most common existing category.

In [ ]:
# Create imputers - median for numeric data, mode / most frequent for categorical data
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

In [ ]:
# Fit imputers using training data only - this aligns with 2d instructions - "fit that treatment using training data only."
numeric_imputer.fit(X_train[numeric_missing])
categorical_imputer.fit(X_train[categorical_missing])

In [ ]:
# View imputation values learned from training data
print("Numeric imputation values:")
for variable, value in zip(numeric_missing, numeric_imputer.statistics_):
    print(variable, ":", value)

print("\nCategorical imputation values:")
for variable, value in zip(categorical_missing, categorical_imputer.statistics_):
    print(variable, ":", value)

In [ ]:
# Verify numeric imputation values against training medians
print("Training medians:")
print(X_train[numeric_missing].median())

print("\nSimpleImputer values:")
print(pd.Series(numeric_imputer.statistics_, index=numeric_missing))

In [ ]:
# Verify categorical imputation values against training modes
print("Training modes:")
print(X_train[categorical_missing].mode().iloc[0])

print("\nSimpleImputer values:")
print(pd.Series(categorical_imputer.statistics_, index=categorical_missing))

In [ ]:
# Apply the imputation values learned from the training data
X_train[numeric_missing] = numeric_imputer.transform(
    X_train[numeric_missing]
)

X_val[numeric_missing] = numeric_imputer.transform(
    X_val[numeric_missing]
)

X_train[categorical_missing] = categorical_imputer.transform(
    X_train[categorical_missing]
)

X_val[categorical_missing] = categorical_imputer.transform(
    X_val[categorical_missing]
)

In [ ]:
# Verify that imputation removed all missing values
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_val:", X_val.isnull().sum().sum())

print("Training observations:", X_train.shape[0])
print("Validation observations:", X_val.shape[0])
print("Total observations:", X_train.shape[0] + X_val.shape[0])

#### Missing Data Treatment and Leakage Verification

Missing values were handled after the training and validation split so that information from the validation data was not used to determine the imputation values. The numeric and categorical imputers were fit using only the training data. The fitted imputers were then applied separately to the training and validation predictors. After imputation, both X_train and X_val contained zero missing values.

This approach prevents data leakage because the validation data did not influence the imputation values used during model development. The validation data remained separate from the fitting of the imputation treatment and was only transformed using values learned from the training data.

### 4a) Primary Model Selection Metric

Recall (sensitivity) for admitted patients will be used as the primary metric for model selection. The hospital's objective is to identify patients likely to be admitted in order to support resource allocation and patient-flow planning. A false negative could result in an actual admission not being anticipated, while a false positive could result in resources being unnecessarily planned for a patient who is not admitted. Because failing to identify an actual admission may have a greater operational consequence, recall is prioritized. This is also important because admitted patients represent a smaller proportion of the dataset.

### 4b) Model Evaluation Framework

All three models will be evaluated on the same validation data using ROC AUC, recall, precision, and F1 score. Recall will remain the primary metric. A classification threshold will be selected using the training data based on the precision-recall tradeoff rather than automatically using the default threshold of 0.50. The same evaluation approach will be used for all three models.

### 5. Model 1 - Random Forest

A Random Forest classifier will be used to predict `Admitted`. Categorical predictors will be encoded using CatBoost encoding within a pipeline. Hyperparameters will be selected using GridSearchCV with stratified cross-validation on the training data only.

In [ ]:
# Identify categorical variables
categorical_vars = X_train.select_dtypes(include='object').columns.tolist()

categorical_vars

#### 5b) Encode categorical variables using CatBoost encoding.

In [ ]:
CatBoostEncoder(cols=categorical_vars, random_state=42)

#### 5c) Hyperparameter Tuning with Grid Search

Grid Search was selected for Random Forest because only two hyperparameters were tuned, with three candidate values for each. This resulted in only nine possible parameter combinations, making it practical to evaluate every combination. Grid Search performs an exhaustive search over the specified parameter combinations, which is appropriate when the search space is relatively small.

In [ ]:
# Set parameters for Grid Search Cross Validation
param_grid = {
    'classifier__max_depth': [5, 10, 15],
    'classifier__n_estimators': [50, 100, 300]
}

RF = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

pipeline = Pipeline([
    ('CatBoostEncoding',
     CatBoostEncoder(cols=categorical_vars, random_state=42)),
    ('classifier', RF)
])

cross_validation = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

Grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='recall',
    cv=cross_validation,
    n_jobs=-1,
    verbose=2
)

In [ ]:
# Fit the Grid Search model on the training data
GS = Grid_search.fit(X_train, y_train)

In [ ]:
# Print best hyperparameters and best cross-validation recall
print("Best parameters:", GS.best_params_)
print("Best cross-validation recall:", GS.best_score_)

In [ ]:
# View Grid Search results
grid_results = pd.DataFrame(GS.cv_results_)[[
    'param_classifier__max_depth',
    'param_classifier__n_estimators',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]].sort_values('rank_test_score')

grid_results

The Grid Search selected `max_depth = 5` and `n_estimators = 50` because this combination produced the highest mean cross-validation recall (0.9862) among the nine parameter combinations evaluated. The results also showed that a maximum depth of 5 performed better than the deeper trees across all tested values of `n_estimators`.

#### 5d) Report the ROC Plot and Model Evaluation Measures

In [ ]:
# Store the best Random Forest model
best_rf = GS.best_estimator_

In [ ]:
# Generate out-of-fold predicted probabilities using the training data
rf_train_prob = cross_val_predict(
    best_rf,
    X_train,
    y_train,
    cv=cross_validation,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

In [ ]:
# Calculate precision and recall across possible thresholds
precision, recall, thresholds = precision_recall_curve(
    y_train,
    rf_train_prob
)

#### F1 Score and Threshold Selection

The F1 score combines precision and recall into a single measure and is useful when both false positives and false negatives are important. It is calculated as the harmonic mean of precision and recall:

F1 = 2 * times (Precision * Recall)/(Precision + Recall)

An F1 score ranges from 0 to 1, with higher values indicating a better balance between precision and recall.

The code below calculates the F1 score across the possible classification thresholds using the out-of-fold predictions from the training data. The threshold with the highest F1 score is selected for the Random Forest model. This allows the classification threshold to be selected using the training data rather than automatically using the default threshold of 0.50 or using the validation data.

In [ ]:
# Calculate F1 score across thresholds to balance precision and recall
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_index = np.argmax(f1_scores)
rf_threshold = thresholds[best_index]

print("Selected threshold:", rf_threshold)
print("Training recall:", recall[best_index])
print("Training precision:", precision[best_index])
print("Training F1 score:", f1_scores[best_index])

In [ ]:
# Generate predicted probabilities on the untouched validation data
rf_val_prob = best_rf.predict_proba(X_val)[:, 1]

# Convert probabilities to predictions using the selected threshold
rf_val_pred = (rf_val_prob >= rf_threshold).astype(int)

In [ ]:
# Calculate Random Forest validation metrics
rf_auc = roc_auc_score(y_val, rf_val_prob)
rf_recall = recall_score(y_val, rf_val_pred)
rf_precision = precision_score(y_val, rf_val_pred)
rf_f1 = f1_score(y_val, rf_val_pred)

print("ROC AUC:", rf_auc)
print("Recall:", rf_recall)
print("Precision:", rf_precision)
print("F1 Score:", rf_f1)

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_val, rf_val_prob)

# Plot ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {rf_auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest ROC Curve')
plt.legend()
plt.show()

#### Random Forest Validation Results

The Random Forest model achieved a ROC AUC of 0.9941 on the untouched validation data. Using the classification threshold of 0.6501 selected from the out-of-fold training predictions, the model achieved a recall of 0.9214, precision of 0.9149, and F1 score of 0.9181.

#### 5e) Explain the Selected Hyperparameters and Evaluate the Model Using the Untouched Validation Data

The Grid Search selected a maximum depth of 5 and 50 estimators because this combination had the highest mean cross-validation recall of 0.9862. A maximum depth of 5 limits how deep each tree can grow, while 50 estimators means the Random Forest combines predictions from 50 trees.

On the untouched validation data, the Random Forest had a ROC AUC of 0.9941, recall of 0.9214, precision of 0.9149, and F1 score of 0.9181. The recall means the model correctly identified 92.1% of the patients who were actually admitted. The precision means about 91.5% of the patients predicted to be admitted were actually admitted. Overall, the model performed well on the validation data while maintaining a good balance between recall and precision.

### 6. Model 2 - LightGBM

#### 6a) Build a LightGBM Classifier to Predict Admitted

In [ ]:
# Create the LightGBM classifier
LM = lgb.LGBMClassifier(
    random_state=42,
    objective='binary',
    boosting_type='gbdt'
)

#### 6b) Encode Categorical Variables Using CatBoost Encoding

In [ ]:
# CatBoost encoding will be included in the LightGBM pipeline
CatBoostEncoder(cols=categorical_vars, random_state=42)

#### 6c) Choose the Best Hyperparameters Using Grid Search

Grid Search was selected for LightGBM because three hyperparameters were tuned across 18 total parameter combinations. This resulted in a relatively small search space, making it practical to evaluate every combination. Grid Search performs an exhaustive search over the specified parameter combinations, which is appropriate when the search space is relatively small.

In [ ]:
# Set a small LightGBM parameter grid based on values used in the lecture demo
param_grid_LM = {
    'classifier__max_depth': [5, 9, 15],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__n_estimators': [300, 800, 1500]
}

# Combine CatBoost encoding and LightGBM so encoding is fit within each CV fold
pipeline_LM = Pipeline([
    ('CatBoostEncoding',
     CatBoostEncoder(cols=categorical_vars, random_state=42)),
    ('classifier', LM)
])

# Use the same cross-validation setup and primary metric as Random Forest
Grid_search_LM = GridSearchCV(
    estimator=pipeline_LM,
    param_grid=param_grid_LM,
    scoring='recall',
    cv=cross_validation,
    n_jobs=-1,
    verbose=2
)

In [ ]:
# Fit the LightGBM Grid Search using the training data
GS_LM = Grid_search_LM.fit(X_train, y_train)

In [ ]:
print("Best parameters:", GS_LM.best_params_)
print("Best CV recall:", GS_LM.best_score_)

In [ ]:
# View Grid Search results
grid_results_LM = pd.DataFrame(GS_LM.cv_results_)[[
    'param_classifier__learning_rate',
    'param_classifier__max_depth',
    'param_classifier__n_estimators',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]].sort_values('rank_test_score')

grid_results_LM

The Grid Search selected a learning rate of 0.05, maximum depth of 5, and 1500 estimators because this combination had the highest mean cross-validation recall of 0.8881 among the parameter combinations tested.

#### 6d) Report the ROC Plot and Model Evaluation Measures

In [ ]:
# Store the best LightGBM model from Grid Search
best_lm = GS_LM.best_estimator_

In [ ]:
# Generate out-of-fold predicted probabilities using the training data
lm_train_prob = cross_val_predict(
    best_lm,
    X_train,
    y_train,
    cv=cross_validation,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

In [ ]:
# Calculate precision and recall across possible thresholds
precision_lm, recall_lm, thresholds_lm = precision_recall_curve(
    y_train,
    lm_train_prob
)

In [ ]:
# Calculate F1 score across thresholds to balance precision and recall
f1_scores_lm = (
    2 * precision_lm[:-1] * recall_lm[:-1]
    / (precision_lm[:-1] + recall_lm[:-1] + 1e-10)
)

best_index_lm = np.argmax(f1_scores_lm)
lm_threshold = thresholds_lm[best_index_lm]

print("Selected threshold:", lm_threshold)
print("Training recall:", recall_lm[best_index_lm])
print("Training precision:", precision_lm[best_index_lm])
print("Training F1 score:", f1_scores_lm[best_index_lm])

In [ ]:
# Generate predicted probabilities on the untouched validation data
lm_val_prob = best_lm.predict_proba(X_val)[:, 1]

# Convert probabilities to predictions using the selected threshold
lm_val_pred = (lm_val_prob >= lm_threshold).astype(int)

In [ ]:
# Calculate LightGBM validation metrics
lm_auc = roc_auc_score(y_val, lm_val_prob)
lm_recall = recall_score(y_val, lm_val_pred)
lm_precision = precision_score(y_val, lm_val_pred)
lm_f1 = f1_score(y_val, lm_val_pred)

print("ROC AUC:", lm_auc)
print("Recall:", lm_recall)
print("Precision:", lm_precision)
print("F1 Score:", lm_f1)

In [ ]:
# Calculate ROC curve
fpr_lm, tpr_lm, thresholds_roc_lm = roc_curve(y_val, lm_val_prob)

# Plot ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr_lm, tpr_lm, label=f'LightGBM (AUC = {lm_auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('LightGBM ROC Curve')
plt.legend()
plt.show()

#### LightGBM Validation Results

The LightGBM model achieved a ROC AUC of 0.9943 on the untouched validation data. Using the classification threshold of 0.8769 selected from the out-of-fold training predictions, the model achieved a recall of 0.8643, precision of 0.9565, and F1 score of 0.9081.

#### 6e) Explain the Selected Hyperparameters, Evaluate the Model Using the Untouched Validation Data, and Compare Its Performance With the Random Forest Model

The Grid Search selected a learning rate of 0.05, maximum depth of 5, and 1500 estimators because this combination had the highest mean cross-validation recall of 0.8881. The lower learning rate makes smaller adjustments during boosting, the maximum depth controls the depth of each tree, and 1500 estimators determines the number of trees used by the model.

On the untouched validation data, LightGBM had a ROC AUC of 0.9943, recall of 0.8643, precision of 0.9565, and F1 score of 0.9081. Compared with Random Forest, LightGBM had a slightly higher ROC AUC (0.9943 vs. 0.9941) and precision (0.9565 vs. 0.9149), while Random Forest had higher recall (0.9214 vs. 0.8643) and F1 score (0.9181 vs. 0.9081). Since recall was selected as the primary metric, Random Forest performed better based on the primary evaluation measure.

### 7. Model 3 - XGBoost

#### 7a) Build an XGBoost Classifier to Predict Admitted

In [ ]:
# Create the XGBoost classifier
XGB = xgb.XGBClassifier(
    random_state=42,
    objective='binary:logistic'
)

#### 7b) Encode Categorical Variables Using CatBoost Encoding

In [ ]:
# CatBoost encoding will be included in the XGBoost pipeline
CatBoostEncoder(cols=categorical_vars, random_state=42)

#### 7c) Choose the Best Hyperparameters Using Random Search

Random Search was selected for XGBoost because several hyperparameters were tuned over a broader range of possible values. An exhaustive Grid Search would require evaluating every possible parameter combination and would become more computationally expensive as the search space increases. Random Search instead evaluates a fixed number of sampled parameter combinations, allowing a broader hyperparameter space to be explored while controlling the computational cost.

In [ ]:
# Define the XGBoost hyperparameter search space
rs_param_grid_XGB = {
    'classifier__max_depth': list(range(3, 12)),
    'classifier__reg_alpha': [0, 0.001, 0.01, 0.1, 1],
    'classifier__subsample': [0.5, 0.75, 1],
    'classifier__learning_rate': np.linspace(0.01, 0.5, 10),
    'classifier__n_estimators': [10, 25, 40]
}

# Combine CatBoost encoding and XGBoost in one pipeline
pipeline_XGB = Pipeline([
    ('CatBoostEncoding', CatBoostEncoder(cols=categorical_vars, random_state=42)),
    ('classifier', XGB)
])

# Use Random Search with the same cross-validation folds and recall scoring
Random_search_XGB = RandomizedSearchCV(
    estimator=pipeline_XGB,
    param_distributions=rs_param_grid_XGB,
    scoring='recall',
    cv=cross_validation,
    n_iter=20,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

RS_XGB = Random_search_XGB.fit(X_train, y_train)

In [ ]:
# Display the best XGBoost hyperparameters and cross-validation recall
print("Best Parameters:", RS_XGB.best_params_)
print("Best CV Recall:", RS_XGB.best_score_)

In [ ]:
# View Random Search results
random_results_XGB = pd.DataFrame(RS_XGB.cv_results_)[[
    'param_classifier__max_depth',
    'param_classifier__reg_alpha',
    'param_classifier__subsample',
    'param_classifier__learning_rate',
    'param_classifier__n_estimators',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]].sort_values('rank_test_score')

random_results_XGB

The Random Search selected a subsample proportion of 0.5, L1 regularization value of 1, 40 estimators, maximum depth of 8, and learning rate of 0.1733 because this combination had the highest mean cross-validation recall of 0.9157 among the parameter combinations tested.

#### 7d) Report the ROC Plot and Model Evaluation Measures

In [ ]:
# Get the best XGBoost model from Random Search
best_XGB = RS_XGB.best_estimator_

# Generate out-of-fold probabilities using only the training data
xgb_oof_prob = cross_val_predict(
    best_XGB,
    X_train,
    y_train,
    cv=cross_validation,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# Calculate precision and recall across possible thresholds
precision_xgb, recall_xgb, thresholds_xgb = precision_recall_curve(
    y_train,
    xgb_oof_prob
)

# Calculate F1 score at each threshold
f1_xgb = 2 * (precision_xgb[:-1] * recall_xgb[:-1]) / (
    precision_xgb[:-1] + recall_xgb[:-1]
)

# Select the threshold with the highest F1 score
best_idx_xgb = np.nanargmax(f1_xgb)
xgb_threshold = thresholds_xgb[best_idx_xgb]

In [ ]:
print("Selected Threshold:", xgb_threshold)
print("Training Recall:", recall_xgb[best_idx_xgb])
print("Training Precision:", precision_xgb[best_idx_xgb])
print("Training F1:", f1_xgb[best_idx_xgb])

In [ ]:
# Generate probabilities on the untouched validation data
xgb_val_prob = best_XGB.predict_proba(X_val)[:, 1]

# Apply the training-based classification threshold
xgb_val_pred = (xgb_val_prob >= xgb_threshold).astype(int)

# Calculate validation metrics
xgb_auc = roc_auc_score(y_val, xgb_val_prob)
xgb_recall = recall_score(y_val, xgb_val_pred)
xgb_precision = precision_score(y_val, xgb_val_pred)
xgb_f1 = f1_score(y_val, xgb_val_pred)

print("Validation ROC AUC:", xgb_auc)
print("Validation Recall:", xgb_recall)
print("Validation Precision:", xgb_precision)
print("Validation F1:", xgb_f1)

In [ ]:
# Calculate ROC curve
fpr_xgb, tpr_xgb, thresholds_roc_xgb = roc_curve(y_val, xgb_val_prob)

# Plot ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {xgb_auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGBoost ROC Curve')
plt.legend()
plt.show()

#### XGBoost Validation Results

The XGBoost model achieved a ROC AUC of 0.9955 on the untouched validation data. Using the classification threshold of 0.5199 selected from the out-of-fold training predictions, the model achieved a recall of 0.9071, precision of 0.9442, and F1 score of 0.9253.

#### 7e) Explain the Selected Hyperparameters, Evaluate the Model Using the Untouched Validation Data, and Compare Its Performance With the Random Forest and LightGBM Models

The Random Search selected a subsample proportion of 0.5, L1 regularization value of 1, 40 estimators, maximum depth of 8, and learning rate of 0.1733 because this combination had the highest mean cross-validation recall of 0.9157 among the parameter combinations tested. The subsample value means 50% of the training observations are sampled for each boosting iteration, the L1 regularization value applies a penalty to help control model complexity, the maximum depth limits the depth of each tree, the learning rate controls the contribution of each new tree, and 40 estimators determines the number of boosting trees used.

On the untouched validation data, XGBoost had a ROC AUC of 0.9955, recall of 0.9071, precision of 0.9442, and F1 score of 0.9253. Compared with Random Forest, XGBoost had a slightly higher ROC AUC (0.9955 vs. 0.9941), precision (0.9442 vs. 0.9149), and F1 score (0.9253 vs. 0.9181), while Random Forest had higher recall (0.9214 vs. 0.9071). Compared with LightGBM, XGBoost had higher ROC AUC (0.9955 vs. 0.9943), recall (0.9071 vs. 0.8643), and F1 score (0.9253 vs. 0.9081), while LightGBM had higher precision (0.9565 vs. 0.9442). Since recall was selected as the primary metric, Random Forest still performed best based on the primary evaluation measure.

#### 7f) Run the SHAP Summary Plot for the Model and Explain the Visual

The lecture applies SHAP directly to a fitted XGBoost model and predictor dataset. In this analysis, XGBoost was fit within a pipeline that includes CatBoost encoding for the categorical variables. Therefore, the fitted CatBoost encoder and XGBoost model must first be separated from the pipeline, and the validation predictors must be transformed using the fitted encoder before calculating SHAP values. This allows the SHAP analysis to use the same fitted model and preprocessing used for the validation predictions.

In [ ]:
# Separate the fitted CatBoost encoder and XGBoost model from the pipeline
xgb_encoder = best_XGB.named_steps['CatBoostEncoding']
xgb_model = best_XGB.named_steps['classifier']

# Apply the fitted CatBoost encoder to the validation predictors
X_val_xgb_encoded = xgb_encoder.transform(X_val)

In [ ]:
# Fit SHAP TreeExplainer to the fitted XGBoost model
xgb_shap_values = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for the encoded validation data
shap_values = xgb_shap_values.shap_values(
    X_val_xgb_encoded,
    approximate=False,
    check_additivity=False
)

In [ ]:
# Create SHAP summary plot
shap.summary_plot(shap_values, X_val_xgb_encoded)

#### SHAP Summary Plot Interpretation

The SHAP summary plot shows that symptom_duration and interest_in_admission had the greatest influence on the XGBoost predictions, followed by OXYGEN_LEVEL and BP_Score. Higher values of symptom_duration and interest_in_admission generally had positive SHAP values, meaning they increased the model's predicted likelihood of admission. Higher OXYGEN_LEVEL values generally had negative SHAP values, while lower oxygen levels increased the predicted likelihood of admission. Higher BP_Score values also generally increased the predicted likelihood of admission.

The features are ordered by their overall importance to the model, with the most influential features appearing at the top. Each point represents an observation, the color represents whether the feature value is relatively high or low, and the SHAP value shows whether that feature increased or decreased the model's prediction of admission. Most of the remaining variables, including the cyclical date and month variables, had SHAP values concentrated closer to zero, indicating that they had less influence on the XGBoost predictions.

### Model Performance Comparison

In [ ]:
# Compare validation performance across all three models
model_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'LightGBM', 'XGBoost'],
    'ROC AUC': [rf_auc, lm_auc, xgb_auc],
    'Recall': [rf_recall, lm_recall, xgb_recall],
    'Precision': [rf_precision, lm_precision, xgb_precision],
    'F1 Score': [rf_f1, lm_f1, xgb_f1]
})

model_comparison.round(4)

The three models all achieved high ROC AUC values above 0.99. Random Forest had the highest recall (0.9214), while XGBoost had the highest ROC AUC (0.9955) and F1 score (0.9253). LightGBM had the highest precision (0.9565) but had the lowest recall (0.8643) and F1 score (0.9081). Since recall was selected as the primary evaluation metric before fitting the models, Random Forest performed best based on the primary metric.